# EDA — Base final sintética de inclusión crediticia

**Objetivo:** documentar la base de entrada final, comprobar que el pipeline vuelve a calcular sus variables derivadas y describir los tres arquetipos y los clusters obtenidos con k=3.

> **Alcance:** los 30.000 registros son sintéticos y fueron construidos con distribuciones intencionales para representar tres perfiles académicos. Las métricas altas muestran que los algoritmos recuperan esa estructura diseñada; no son evidencia de separación en clientes reales ni de validez crediticia. `arquetipo_referencia` se conserva en un archivo de auditoría aparte y no es predictor del clustering.

## 1. Carga y estructura

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(obj): print(obj)

ROOT = Path.cwd()
if not (ROOT / 'data/processed/perfiles_crediticios_procesados.csv').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data/processed/perfiles_crediticios_procesados.csv'
REFERENCE = ROOT / 'data/processed/referencia_arquetipos_sinteticos.csv'
METRICS = ROOT / 'data/processed/comparacion_clusterizaciones.csv'
ASSIGNMENTS = ROOT / 'data/processed/perfiles_clusterizados.csv'
df = pd.read_csv(DATA)
reference = pd.read_csv(REFERENCE)
metrics = pd.read_csv(METRICS)
assignments = pd.read_csv(ASSIGNMENTS)
assert df['id_persona'].equals(reference['id_persona'])
print(f'{len(df):,} perfiles | {df.shape[1]} columnas procesadas')
print('Archivo:', DATA)
display(df.head())

## 2. Diccionario por etapa y controles de calidad

La entrada contiene los campos reportados por el perfil ficticio. El pipeline calcula gastos totales, flujo, ratios, ingreso per cápita, cobertura de ahorro, proxies y segmento inferido.

In [ ]:
from src.pipeline.perfil_financiero import calcular_variables_derivadas

DERIVED = [
    'nivel_educacion_financiera', 'gastos_esenciales_mensuales_cop',
    'flujo_mensual_disponible_cop', 'ratio_deuda_ingreso', 'ratio_gastos_ingreso',
    'ingreso_per_capita_cop', 'cobertura_ahorro_meses',
    'capacidad_cuota_referencial_cop', 'nivel_experiencia_crediticia_proxy',
    'segmento_generador',
]
sample_audit = df.sample(n=min(1000, len(df)), random_state=42)
recalculated = pd.DataFrame(
    [calcular_variables_derivadas(row) for row in sample_audit.to_dict(orient='records')],
    index=sample_audit.index,
)
checks = {
    'IDs duplicados': int(df.id_persona.duplicated().sum()),
    'celdas faltantes': int(df.isna().sum().sum()),
    'edades fuera de 18–74': int(((df.edad < 18) | (df.edad > 74)).sum()),
    'campos derivados ausentes': len(set(DERIVED) - set(df.columns)),
    'ratios vacíos esperados por ingreso cero': int((df.ingreso_neto_mensual_cop.eq(0) & df.ratio_deuda_ingreso.isna()).sum() + (df.ingreso_neto_mensual_cop.eq(0) & df.ratio_gastos_ingreso.isna()).sum()),
}
numeric_derived = [c for c in DERIVED if pd.api.types.is_numeric_dtype(df[c])]
categorical_derived = [c for c in DERIVED if c not in numeric_derived]
numeric_match = np.isclose(sample_audit[numeric_derived].to_numpy(dtype=float), recalculated[numeric_derived].to_numpy(dtype=float), rtol=0, atol=1e-8, equal_nan=True).all(axis=1)
categorical_match = sample_audit[categorical_derived].fillna('__NA__').astype(str).eq(recalculated[categorical_derived].fillna('__NA__').astype(str)).all(axis=1)
checks['perfiles auditados recalculando derivados'] = len(sample_audit)
checks['derivados distintos al recalcular pipeline en la muestra'] = int((~(numeric_match & categorical_match)).sum())
checks['ratios no definidos con ingreso positivo'] = int(((df.ingreso_neto_mensual_cop > 0) & (df.ratio_deuda_ingreso.isna() | df.ratio_gastos_ingreso.isna())).sum())
checks['referencias sin correspondencia de ID'] = int((df.id_persona != reference.id_persona).sum())
display(pd.Series(checks, name='hallazgos'))
display(pd.DataFrame({'tipo': ['entrada'] * (df.shape[1] - len(DERIVED)) + ['derivada'] * len(DERIVED), 'campo': [c for c in df.columns if c not in DERIVED] + DERIVED}))

## 3. Composición de los arquetipos diseñados

Las proporciones objetivo son 40% viabilidad formal, 30% cuidado del historial y 30% inicio en educación financiera. Se muestran para describir el diseño del generador, no como estimación de prevalencia.

In [ ]:
labels = {
    0: 'Viabilidad estudio formal',
    1: 'Cuidar historial crediticio',
    2: 'Iniciar educación financiera',
}
reference['arquetipo'] = reference.arquetipo_referencia.map(labels)
composition = reference.arquetipo.value_counts().rename('perfiles').to_frame()
composition['porcentaje'] = (composition.perfiles / len(reference) * 100).round(1)
display(composition)
ax = composition.porcentaje.sort_values().plot.barh(color='#287C8E', figsize=(9, 4))
ax.set(title='Distribución objetivo de la base sintética', xlabel='% de perfiles', ylabel='')
ax.grid(axis='x', alpha=.2)
plt.tight_layout(); plt.show()

## 4. Variables financieras por arquetipo

Las distribuciones financieras se definieron deliberadamente para que los perfiles tengan patrones distinguibles. Las variables del hogar y la edad se conservaron sin asociarlas a las clases objetivo.

In [ ]:
plot_df = df.merge(reference[['id_persona', 'arquetipo']], on='id_persona', validate='one_to_one')
features = [
    ('meses_historial_crediticio', 'Meses de historial'),
    ('moras_ultimos_12_meses', 'Moras últimos 12 meses'),
    ('ratio_gastos_ingreso', 'Gastos / ingreso'),
    ('ratio_deuda_ingreso', 'Cuotas de deuda / ingreso'),
    ('cobertura_ahorro_meses', 'Cobertura de ahorro (meses)'),
]
fig, axes = plt.subplots(1, len(features), figsize=(17, 4))
for ax, (field, title) in zip(axes, features):
    series = [plot_df.loc[plot_df.arquetipo == name, field].dropna().to_numpy() for name in labels.values()]
    ax.boxplot(series, showfliers=False)
    ax.set(title=title, xticks=range(1, 4), xticklabels=['Formal', 'Cuidar', 'Educación'], ylabel='')
    ax.tick_params(axis='x', labelrotation=25)
    ax.grid(axis='y', alpha=.2)
plt.tight_layout(); plt.show()

summary_fields = ['ingreso_per_capita_cop', 'ratio_gastos_ingreso', 'ratio_deuda_ingreso', 'cobertura_ahorro_meses', 'meses_historial_crediticio', 'moras_ultimos_12_meses', 'flujo_mensual_disponible_cop']
display(plot_df.groupby('arquetipo')[summary_fields].median().round(2))

## 5. Resultado de los tres modelos con k=3

In [ ]:
display(metrics[['algoritmo', 'k', 'silhouette_gower', 'menor_cluster_pct', 'mayor_cluster_pct', 'ari_vs_arquetipo_referencia']].round(4))
winner = assignments[['id_persona', 'cluster']]
cluster_table = reference[['id_persona', 'arquetipo']].merge(winner, on='id_persona', validate='one_to_one')
cross = pd.crosstab(cluster_table.arquetipo, cluster_table.cluster, normalize='index').mul(100).round(1)
display(cross)
gower_cross = pd.crosstab(cluster_table.arquetipo, assignments.cluster_gower_k_medoids, normalize='index').mul(100).round(1)
print('Correspondencia arquetipo–cluster de Gower + K-Medoids:')
display(gower_cross)
print('Algoritmo guardado en las asignaciones:', assignments.algoritmo_seleccionado.iloc[0])
print('Frecuencia de clusters en los 30.000 perfiles:')
display(winner.cluster.value_counts().sort_index().rename('perfiles').to_frame().assign(porcentaje=lambda x: (x.perfiles / len(winner) * 100).round(1)))

## 6. Conclusiones y límites

- El pipeline recalcula todos los campos derivados de manera reproducible; el control compara los resultados con los guardados en la base.
- Los tres métodos, con k=3, recuperan los patrones usados para construir estos perfiles. One-Hot + K-Means y K-Prototypes quedan prácticamente empatados en Silhouette Gower (≈0,417); Gower + K-Medoids obtiene ≈0,413.
- La referencia de arquetipo no forma parte de las entradas del clustering. Su acuerdo alto se debe al diseño intencional de las distribuciones, no a validación externa.
- `segmento_generador` es una regla derivada independiente y no se usa para entrenar los clusters.
- Los resultados son didácticos con datos sintéticos; no deben interpretarse como prevalencias poblacionales ni como decisiones reales de crédito.